# Text Classification with BERT and Cross-Validation

This notebook implements a text sequence classification model based on BERT with 5-fold cross-validation. The pipeline includes training the model, evaluating it, and saving/loading the best model.

In [1]:
# Install necessary libraries
!pip install torch transformers datasets scikit-learn matplotlib pandas tqdm colorama seaborn ipywidgets
!python -m spacy download ru_core_news_sm


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 31.5 MB/s eta 0:00:00 MB/s eta 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


## Import Libraries

In [2]:
import os
import re
import nltk
import spacy
import torch
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from colorama import Fore, Style
from nltk.corpus import stopwords
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    accuracy_score,
    roc_auc_score, 
    ConfusionMatrixDisplay
)
from transformers import BertForSequenceClassification, AutoTokenizer, get_scheduler
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import Dataset

warnings.simplefilter(action='ignore', category=FutureWarning)

## Data Preprocessing

In [3]:
# Loading the data
file_path = "all_channel_posts.csv"  # Provide the path to your CSV file
df = pd.read_csv('all_channel_posts.csv', parse_dates=['date'])

df = df.drop_duplicates(subset=['text'])
df = df[df['category'] != 'other']
df = df.reset_index(drop=True)

In [4]:
nltk.download('stopwords')

stop = set(stopwords.words('russian'))
nlp = spacy.load('ru_core_news_sm')

def clean_text_symbols(text: str) -> str:
    text = re.sub(r"(https?://|www\.)\S+|bit\.ly/\S+|t\.co/\S+", "", text)  
    text = re.sub(r"#[\wа-яА-ЯёЁ]+", "", text)  
    text = re.sub(r"@[\wа-яА-ЯёЁ]+", "", text)  
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"(\*\*|__)(.*?)\1", r"\2", text)
    text = re.sub(r"(\*|_)(.*?)\1", r"\2", text)
    text = re.sub(r"~~(.*?)~~", r"\1", text)
    text = re.sub(r"`{1,2}(.*?)`{1,2}", r"\1", text)
    text = re.sub(r"#{1,6}\s*", "", text)
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)
    text = re.sub(r">\s*", "", text)
    text = re.sub(r"[^a-zA-Zа-яА-ЯёЁ0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stop_words(text: str, stop: set) -> str:
    text = " ".join([word for word in text.split() if word not in stop])
    return text

def lemmatize_text(text: str) -> str:
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc]
    return ' '.join(lemmas) 

def preprocess_text(text: str, stop: set) -> str:
    text = text.lower()
    text = clean_text_symbols(text)
    text = lemmatize_text(text)
    text = remove_stop_words(text, stop)
    return text

df['clean_text'] = df['text'].apply(lambda x: preprocess_text(x, stop))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nktkln/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
def get_last_text_line(text: str) -> str:
    if len(text.split('\n')) == 1:
        return ""
    return text.strip().split('\n')[-1]

def remove_frequent_last_line(text: str, frequent_lines: list) -> str:
    lines = text.strip().split('\n')
    if len(lines) > 1 and lines[-1] in frequent_lines:
        lines = lines[:-1]
    return '\n'.join(lines).strip()

last_lines_series = df['clean_text'].apply(get_last_text_line)
last_lines_series = last_lines_series.replace("", pd.NA).dropna()
last_line_frequencies = last_lines_series.value_counts()
frequent_last_lines = last_line_frequencies[last_line_frequencies > 1].index.tolist()

df['clean_text'] = df['clean_text'].apply(lambda x: remove_frequent_last_line(x, frequent_last_lines))

In [6]:
stop.update(["это", "также", "подписаться", "который", "изза", "прислать", "ещё", "которые", "топор", "подписывайся", "mash", "shot", "live", "новостную", "кибертопор"])

df['clean_text'] = df['clean_text'].apply(lambda x: remove_stop_words(x, stop))

In [7]:
bigrams_to_remove = {
    "стало известно", 
    "данный момент", 
    "читать статью", 
    "читать vpn",
    "риа новости"
}

def remove_bigrams(text: str, bigrams_to_remove: set) -> str:
    for bigram in bigrams_to_remove:
        if bigram in text:
            text = text.replace(bigram, "")
    return text.strip()

df['clean_text'] = df['clean_text'].apply(lambda x: remove_bigrams(x, bigrams_to_remove))

In [8]:
df['hour'] = df['date'].dt.hour # <-

In [9]:
# Extracting text and category labels
texts = ('[TEXT] ' + df['clean_text'] + ' [HOUR] ' + df['hour'].astype(str)).tolist()
labels = df['category'].tolist()

# Converting labels to numerical identifiers
# unique_labels = list(set(labels)).sort()
unique_labels = ['business', 'it', 'personal', 'weather', 'gaming', 'finances', 'stuff', 'political', 'advertisement', 'science', 'moscow']
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
labels = [label_to_id[label] for label in labels]

# Splitting data into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

## Cross-Validation Setup

In [10]:
# Setting up cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loading the pre-trained tokenizer and setting up the device
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Tokenization Function

In [11]:
# Tokenization function
def tokenize_function(examples: dict) -> dict:
    """
    Tokenizes input texts using the pre-trained tokenizer.

    :param examples: A dictionary containing the texts to be tokenized.
    :returns: A dictionary containing tokenized texts.
    """
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

## Data Preparation for Training and Validation

In [12]:
# Preparing data for training and validation
def prepare_data(texts: list, labels: list, fold_idx: int, kfold: StratifiedKFold) -> tuple:
    """
    Prepares the training and validation datasets for the given fold iteration.

    :param texts: List of input texts.
    :param labels: List of labels.
    :param fold_idx: Index of the current fold.
    :param kfold: StratifiedKFold object for cross-validation.
    :returns: A tuple containing:
            - train_data: A Dataset object for the training data.
            - val_data: A Dataset object for the validation data.
    """
    train_idx, val_idx = list(kfold.split(texts, labels))[fold_idx]

    train_texts = [texts[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_texts = [texts[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]

    train_data = Dataset.from_dict({"text": train_texts, "label": train_labels})
    val_data = Dataset.from_dict({"text": val_texts, "label": val_labels})

    train_data = train_data.map(tokenize_function, batched=True)
    val_data = val_data.map(tokenize_function, batched=True)

    train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    return train_data, val_data

## Model Training Function

In [13]:
# Model training function
def train_model(model: torch.nn.Module, train_loader: DataLoader, optimizer: torch.optim.Optimizer, scheduler: torch.optim.lr_scheduler.LambdaLR) -> float:
    """
    Trains the model for one epoch.

    :param model: The model to be trained.
    :param train_loader: DataLoader for the training set.
    :param optimizer: The optimizer for the model.
    :param scheduler: Learning rate scheduler.
    :returns: The average loss for the epoch.
    """
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc="Обучение"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    return avg_loss

## Model Evaluation Function

In [14]:
# Model evaluation function
def evaluate_model(model: torch.nn.Module, val_loader: DataLoader, device: torch.device) -> tuple:
    """
    Evaluates the model on validation data.

    :param model: The model to be evaluated.
    :param val_loader: DataLoader for the validation set.
    :param device: The device (CPU or GPU) on which the model is run.
    :returns: Contains three elements:
            - all_preds: List of predicted labels.
            - all_labels: List of true labels.
            - all_probs: List of predicted probabilities for each class.
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Оценка"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(outputs.logits, axis=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return all_preds, all_labels, all_probs

## Preparing Test Data

In [15]:
# Test data preparation function
def prepare_test_data(texts: list, labels: list) -> Dataset:
    """
    Prepares the test dataset for evaluation.

    :param texts: List of test texts.
    :param labels: List of test labels.
    :returns: The test dataset in a format suitable for DataLoader.
    """
    test_data = Dataset.from_dict({"text": texts, "label": labels})
    test_data = test_data.map(tokenize_function, batched=True)
    test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return test_data

## Saving and Loading the Best Model

In [16]:
# Function to save the best model
def save_model(model_path: str, model: torch.nn.Module, tokenizer: AutoTokenizer) -> None:
    """
    Saves the model and tokenizer to disk.

    :param model_path: Path to the directory where the model and tokenizer will be saved.
    :param model: The trained model.
    :param tokenizer: The tokenizer used with the model.
    """
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"{Fore.GREEN}Model saved at {Style.RESET_ALL}{model_path}")

In [17]:
# Function to load the best model
def load_model(model_path: str) -> tuple:
    """
    Loads the saved best model and tokenizer from the specified path.

    :param model_path: Path to the directory containing the saved model and tokenizer.
    :returns: The model and tokenizer, if available, otherwise None.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path {model_path} does not exist.")
    
    model = BertForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print(f"{Fore.GREEN}Model and tokenizer loaded from {Style.RESET_ALL}{model_path}")
    return model, tokenizer

## Main Training Loop

In [18]:
# Training parameters
epochs = 3
batch_size = 8
learning_rate = 5e-5

# Global variables
best_accuracy = 0.0

# Training loop for each fold in cross-validation
for fold_idx in range(kfold.n_splits):
    print(f"\n{Fore.CYAN}Training fold {Style.RESET_ALL}{fold_idx + 1}")

    # Preparing data for training and validation
    train_data, val_data = prepare_data(train_texts, train_labels, fold_idx, kfold)

    # Loading the model for each fold
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=12)
    model.to(device)

    # Setting up optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    num_training_steps = epochs * len(train_data) // batch_size
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    # DataLoader for training and validation
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)

    # Training the model over several epochs
    for epoch in range(epochs):
        print(f"{Fore.YELLOW}Epoch {Style.RESET_ALL}{epoch + 1}/{epochs}")
        avg_loss = train_model(model, train_loader, optimizer, lr_scheduler)
        print(f"{Fore.RED}Training loss: {Style.RESET_ALL}{avg_loss:.4f}")

    # Evaluating accuracy on validation data
    all_preds, all_labels, _ = evaluate_model(model, val_loader, device)
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"{Fore.MAGENTA}Validation Accuracy: {Style.RESET_ALL}{accuracy:.4f}")

    # Saving the model if accuracy improved
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        save_model("best_model", model, tokenizer)

# Clearing CUDA cache if using GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Training fold 1


Map:   0%|          | 0/13038 [00:00<?, ? examples/s]

Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3


...

Epoch 1/3


Обучение: 100%|██████████| 1630/1630 [29:43<00:00,  1.09s/it]


Training loss: 1.7567
Epoch 2/3


Обучение:  61%|██████    | 998/1630 [19:59<12:39,  1.20s/it]


KeyboardInterrupt: 

## Final Evaluation

In [ ]:
# Loading the best model after training
model, tokenizer = load_model("best_model")
model.to(device)

# Preparing the test dataset for final evaluation
test_data = prepare_test_data(test_texts, test_labels)
test_loader = DataLoader(test_data, batch_size=batch_size)

# Final evaluation on the test set
all_preds, all_labels, all_probs = evaluate_model(model, test_loader, device)

## Results and Metrics

In [ ]:
# Printing final evaluation results
accuracy = accuracy_score(all_labels, all_preds)
print(f"{Fore.MAGENTA}Test Accuracy: {Style.RESET_ALL}{accuracy:.4f}\n")

# Unique classes present in the test set
present_classes = sorted(set(all_labels))

# Printing classification report
print(f"{Fore.MAGENTA}Classification Report:{Style.RESET_ALL}")
print(classification_report(
    all_labels,
    all_preds,
    labels=present_classes,
    target_names=[id_to_label[idx] for idx in present_classes]
))

if len(set(all_labels)) == len(set(all_preds)):
    all_labels_ovr = np.zeros((len(all_labels), len(unique_labels)))
    for i, label in enumerate(all_labels):
        all_labels_ovr[i][label] = 1

    all_probs = np.array(all_probs)
    roc_auc = roc_auc_score(all_labels_ovr, all_probs, average="macro", multi_class="ovr")
    print(f"{Fore.MAGENTA}Macro-Averaged ROC-AUC: {Style.RESET_ALL}{roc_auc:.4f}")

In [ ]:
# Generating and displaying the confusion matrix
present_classes = sorted(set(all_labels).union(set(all_preds)))
cm = confusion_matrix(all_labels, all_preds, labels=present_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[id_to_label[idx] for idx in present_classes])

disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()